In [ ]:
# =====================================================================
# CELL 1: DEPENDENCIES & TRAINING PIPELINE
# =====================================================================
# !pip install -q torch transformers datasets trl peft accelerate bitsandbytes

import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# 1. PARSE CHAT DATASET FROM TXT FILE
file_path = "./training_data/chat-data.txt"
print(f"Reading chat training data from: {file_path}")

with open(file_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

# Split chunks by '---' divider
chunks = [chunk.strip() for chunk in raw_content.split("---") if chunk.strip()]
dataset = Dataset.from_dict({"text": chunks})
print(f"Successfully loaded {len(dataset)} chat blocks.")

# 2. LOAD BASE MODEL & TOKENIZER
model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # Ideal lightweight desktop/mobile base
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# 3. LORA CONFIGURATION
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = TrainingArguments(
    output_dir="./omoji_chat_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=5,
    fp16=True,
    optim="paged_adamw_8bit"
)

# 4. SFT TRAINER EXECUTION
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args
)

print("Starting local chat fine-tuning...")
trainer.train()

# 5. SAVE LORA ADAPTER
adapter_path = "./omoji_chat_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# 6. MERGE ADAPTER WITH BASE MODEL FOR STANDALONE EXPORT
print("Merging LoRA weights back into base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="cpu"
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

final_hf_path = "./omoji_chat_hf_merged"
merged_model.save_pretrained(final_hf_path)
tokenizer.save_pretrained(final_hf_path)
print(f"Merged model saved successfully to: '{final_hf_path}'")

In [ ]:
# --- BLOCK 2: ONNX MODEL CONVERSION ---
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = "./mobile_model_raw"
onnx_output_path = "./mobile_model_raw/omoji_classifier.onnx"

# 1. Load trained model & tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model.eval()

# 2. Create dummy input matching mobile expected shape (batch_size, sequence_length)
dummy_text = "Sample mobile test prompt"
dummy_input = tokenizer(
    dummy_text, 
    return_tensors="pt", 
    padding="max_length", 
    max_length=64, 
    truncation=True
)

# 3. Export to ONNX format
torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    onnx_output_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14
)

print(f"Successfully exported mobile-ready ONNX model to: {onnx_output_path}")

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# 1. PARSE CHAT DATASET
project_root = "/home/uninterested_k/Documents/Omoji"
file_path = os.path.join(project_root, "lib", "models", "training data", "chat-data.txt")
print(f"Reading chat training data from: {file_path}")

with open(file_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

chunks = [chunk.strip() for chunk in raw_content.split("---") if chunk.strip()]
dataset = Dataset.from_dict({"text": chunks})

# 2. LOAD BASE MODEL & TOKENIZER (Standard FP16, no bitsandbytes needed!)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. LORA CONFIGURATION & TRAINING ARGS
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = TrainingArguments(
    output_dir=os.path.join(project_root, "omoji_chat_results"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=5,
    fp16=True,
    optim="adamw_torch" # Standard optimizer since we aren't using paged 8-bit
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args
)

print("Starting local chat fine-tuning...")
trainer.train()

# 4. SAVE ADAPTER & MERGE WEIGHTS
adapter_path = os.path.join(project_root, "omoji_chat_adapter")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("Merging LoRA weights back into base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="cpu"
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

# 5. SAVE FINAL MERGED MODEL TO PROJECT ROOT DIRECTORY
final_hf_path = os.path.join(project_root, "omoji_chat_hf_merged")
merged_model.save_pretrained(final_hf_path)
tokenizer.save_pretrained(final_hf_path)
print(f"Merged model saved successfully to: '{final_hf_path}'")